# FraudLens Data Quality Checks

Use this notebook after regeneration or benchmark processing to answer a few focused questions:

- Do Raw, Bronze, Silver, and Gold row counts match?
- Do labels align with the Gold features we expect?
- Are the worst outliers understandable for the current dataset mode?


In [ ]:
from pathlib import Path
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Choose "synthetic" or "sparkov".
DATASET_MODE = "synthetic"

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root.parent != project_root:
    project_root = project_root.parent

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_lens.ingest import load_paths_config, load_sparkov_config

spark = (
    SparkSession.builder.appName("FraudLens-Data-Quality")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

default_cfg = load_paths_config().get("data", {})
sparkov_cfg = load_sparkov_config().get("sparkov", {})

if DATASET_MODE == "sparkov":
    raw_path = (project_root / sparkov_cfg.get("normalized_raw_path", "data/raw_sparkov")).resolve()
    raw_glob = "*.json"
    bronze_path = (project_root / sparkov_cfg.get("bronze_path", "data/benchmark/bronze_sparkov")).resolve()
    silver_path = (project_root / sparkov_cfg.get("silver_path", "data/benchmark/silver_sparkov")).resolve()
    gold_path = (project_root / sparkov_cfg.get("gold_path", "data/benchmark/gold_sparkov")).resolve()
else:
    raw_path = (project_root / default_cfg.get("raw", "data/raw")).resolve()
    raw_glob = "*.jsonl"
    bronze_path = (project_root / default_cfg.get("bronze", "data/bronze")).resolve()
    silver_path = (project_root / default_cfg.get("silver", "data/silver")).resolve()
    gold_path = (project_root / default_cfg.get("gold", "data/gold")).resolve()

print(f"Dataset mode: {DATASET_MODE}")


In [ ]:
# Load the current pipeline outputs.

raw_df = spark.read.json(str(raw_path / raw_glob))
bronze_df = spark.read.parquet(str(bronze_path))
silver_df = spark.read.parquet(str(silver_path))
gold_df = spark.read.parquet(str(gold_path))

print("Loaded layers:")
print(f"  Raw:    {raw_path}")
print(f"  Bronze: {bronze_path}")
print(f"  Silver: {silver_path}")
print(f"  Gold:   {gold_path}")


In [ ]:
# Row counts and schema checkpoints.

layer_counts = [
    ("raw", raw_df.count()),
    ("bronze", bronze_df.count()),
    ("silver", silver_df.count()),
    ("gold", gold_df.count()),
]

print("Row counts:")
for layer, count in layer_counts:
    print(f"  {layer:<6} {count:,}")

print("\nSilver schema:")
silver_df.printSchema()

print("Gold schema:")
gold_df.printSchema()


In [ ]:
# Compact label coverage summary.

coverage = gold_df.agg(
    F.count("*").alias("gold_rows"),
    F.sum(F.col("ref_transaction_id").isNotNull().cast("int")).alias("rows_with_ref_transaction_id"),
    F.sum((F.col("anomaly_type") == "impossible_travel").cast("int")).alias("impossible_travel_rows"),
    F.sum((F.col("anomaly_type") == "spending_spike").cast("int")).alias("spending_spike_rows"),
    F.sum((F.col("anomaly_type") == "fraud").cast("int")).alias("fraud_rows"),
    F.sum(F.col("speed_from_prev_kmh").isNull().cast("int")).alias("rows_with_null_speed"),
).first()

print("Coverage summary:")
for key in coverage.asDict():
    print(f"  {key}: {coverage[key]:,}")

print("\nAnomaly distribution:")
gold_df.groupBy("anomaly_type").count().orderBy("anomaly_type").show(truncate=False)


## Rule Alignment

These checks adapt to the current dataset mode:

- synthetic mode keeps explicit impossible-travel and spending-spike checks
- Sparkov mode treats `fraud` as a generic benchmark label and avoids assuming it is a travel-specific label


In [ ]:
# Amount-based checks.

spike_threshold = 3.0

if DATASET_MODE == "sparkov":
    amount_quality = (
        gold_df.select(
            (F.col("anomaly_type") == "fraud").alias("is_fraud_label"),
            (F.col("amount_zscore") >= spike_threshold).alias("amount_flag"),
        )
        .groupBy("is_fraud_label")
        .agg(
            F.count("*").alias("rows"),
            F.sum(F.col("amount_flag").cast("int")).alias("rows_above_amount_threshold"),
        )
        .withColumn("frac_above_threshold", F.round(F.col("rows_above_amount_threshold") / F.col("rows"), 4))
        .orderBy("is_fraud_label")
    )
    print(f"Benchmark amount summary (amount_zscore >= {spike_threshold}):")
    amount_quality.show(truncate=False)
    print("Top non-fraud rows by amount_zscore:")
    gold_df.where(F.col("anomaly_type") != "fraud").orderBy(F.desc("amount_zscore")).select(
        "transaction_id", "card_id", "amount", "amount_zscore", "anomaly_type"
    ).show(20, truncate=False)
else:
    spike_quality = (
        gold_df.select(
            "anomaly_type",
            (F.col("amount_zscore") >= spike_threshold).alias("spike_by_rule"),
        )
        .groupBy("anomaly_type")
        .agg(
            F.count("*").alias("rows"),
            F.sum(F.col("spike_by_rule").cast("int")).alias("spike_by_rule_rows"),
        )
        .withColumn("spike_by_rule_frac", F.round(F.col("spike_by_rule_rows") / F.col("rows"), 4))
        .orderBy("anomaly_type")
    )
    print(f"Spending spike rule summary (amount_zscore >= {spike_threshold}):")
    spike_quality.show(truncate=False)
    print("Top non-spike rows by amount_zscore:")
    gold_df.where(F.col("anomaly_type") != "spending_spike").orderBy(F.desc("amount_zscore")).select(
        "transaction_id", "card_id", "amount", "amount_zscore", "anomaly_type"
    ).show(20, truncate=False)


In [ ]:
# Compact distribution views.

import matplotlib.pyplot as plt

plot_df = gold_df.select("anomaly_type", "amount_zscore", "speed_from_prev_kmh")

if DATASET_MODE == "sparkov":
    travel_plot_pdf = (
        plot_df.where(F.col("speed_from_prev_kmh").isNotNull())
        .withColumn(
            "travel_label",
            F.when(F.col("anomaly_type") == "fraud", F.lit("fraud")).otherwise(F.lit("non_fraud")),
        )
        .sampleBy("travel_label", fractions={"fraud": 1.0, "non_fraud": 0.03}, seed=42)
        .toPandas()
    )
    amount_plot_pdf = (
        plot_df.where(F.col("amount_zscore").isNotNull())
        .withColumn(
            "amount_label",
            F.when(F.col("anomaly_type") == "fraud", F.lit("fraud")).otherwise(F.lit("non_fraud")),
        )
        .sampleBy("amount_label", fractions={"fraud": 1.0, "non_fraud": 0.03}, seed=43)
        .toPandas()
    )
    left_title = "Travel Speed Distribution by Fraud Label"
    right_title = "Amount Z-Score Distribution by Fraud Label"
    left_labels = ["non_fraud", "fraud"]
    right_labels = ["non_fraud", "fraud"]
    left_col = "travel_label"
    right_col = "amount_label"
    left_colors = {"non_fraud": "#4c78a8", "fraud": "#e45756"}
    right_colors = {"non_fraud": "#72b7b2", "fraud": "#f58518"}
else:
    travel_plot_pdf = (
        plot_df.where(F.col("speed_from_prev_kmh").isNotNull())
        .withColumn(
            "travel_label",
            F.when(F.col("anomaly_type") == "impossible_travel", F.lit("impossible_travel")).otherwise(F.lit("non_travel")),
        )
        .sampleBy("travel_label", fractions={"impossible_travel": 1.0, "non_travel": 0.03}, seed=42)
        .toPandas()
    )
    amount_plot_pdf = (
        plot_df.where(F.col("amount_zscore").isNotNull())
        .withColumn(
            "amount_label",
            F.when(F.col("anomaly_type") == "spending_spike", F.lit("spending_spike")).otherwise(F.lit("non_spike")),
        )
        .sampleBy("amount_label", fractions={"spending_spike": 1.0, "non_spike": 0.03}, seed=43)
        .toPandas()
    )
    left_title = "Travel Speed Distribution"
    right_title = "Amount Z-Score Distribution"
    left_labels = ["non_travel", "impossible_travel"]
    right_labels = ["non_spike", "spending_spike"]
    left_col = "travel_label"
    right_col = "amount_label"
    left_colors = {"non_travel": "#4c78a8", "impossible_travel": "#e45756"}
    right_colors = {"non_spike": "#72b7b2", "spending_spike": "#f58518"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label in left_labels:
    subset = travel_plot_pdf[travel_plot_pdf[left_col] == label]
    if not subset.empty:
        axes[0].hist(subset["speed_from_prev_kmh"], bins=50, alpha=0.55, label=label, color=left_colors[label])
for threshold in [250, 500, 1000]:
    axes[0].axvline(threshold, color="black", linestyle="--", linewidth=1)
axes[0].set_title(left_title)
axes[0].set_xlabel("speed_from_prev_kmh")
axes[0].set_ylabel("count")
axes[0].legend()

for label in right_labels:
    subset = amount_plot_pdf[amount_plot_pdf[right_col] == label]
    if not subset.empty:
        axes[1].hist(subset["amount_zscore"], bins=50, alpha=0.55, label=label, color=right_colors[label])
axes[1].axvline(3.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title(right_title)
axes[1].set_xlabel("amount_zscore")
axes[1].set_ylabel("count")
axes[1].legend()

fig.tight_layout()
plt.show()


In [ ]:
# Travel-oriented diagnostics.

speed_thresholds = [250.0, 500.0, 1000.0]

if DATASET_MODE == "sparkov":
    fraud_df = gold_df.where(F.col("anomaly_type") == "fraud")
    non_fraud_df = gold_df.where(F.col("anomaly_type") != "fraud")
    print("Benchmark speed summary by fraud label:")
    fraud_df.agg(
        F.count("*").alias("fraud_rows"),
        F.sum(F.col("speed_from_prev_kmh").isNull().cast("int")).alias("fraud_rows_with_null_speed"),
        F.min("speed_from_prev_kmh").alias("min_speed"),
        F.expr("percentile_approx(speed_from_prev_kmh, 0.5)").alias("median_speed"),
        F.max("speed_from_prev_kmh").alias("max_speed"),
    ).show(truncate=False)
    threshold_rows = []
    for threshold in speed_thresholds:
        threshold_rows.append((
            threshold,
            fraud_df.where(F.col("speed_from_prev_kmh") >= threshold).count(),
            non_fraud_df.where(F.col("speed_from_prev_kmh") >= threshold).count(),
        ))
    threshold_summary = spark.createDataFrame(
        threshold_rows,
        ["speed_threshold_kmh", "fraud_rows_above_threshold", "non_fraud_rows_above_threshold"],
    )
    print("Threshold sweep:")
    threshold_summary.show(truncate=False)
    print("Fastest non-fraud rows:")
    non_fraud_df.where(F.col("speed_from_prev_kmh").isNotNull()).orderBy(F.desc("speed_from_prev_kmh")).select(
        "transaction_id", "card_id", "event_time", "distance_from_prev_km", "hours_since_prev", "speed_from_prev_kmh", "anomaly_type"
    ).show(20, truncate=False)
else:
    travel_df = gold_df.where(F.col("anomaly_type") == "impossible_travel")
    non_travel_df = gold_df.where(F.col("anomaly_type") != "impossible_travel")
    print("Impossible-travel speed summary:")
    travel_df.agg(
        F.count("*").alias("travel_rows"),
        F.sum(F.col("ref_transaction_id").isNotNull().cast("int")).alias("travel_rows_with_ref"),
        F.sum(F.col("speed_from_prev_kmh").isNull().cast("int")).alias("travel_rows_with_null_speed"),
        F.min("speed_from_prev_kmh").alias("min_speed"),
        F.expr("percentile_approx(speed_from_prev_kmh, 0.5)").alias("median_speed"),
        F.max("speed_from_prev_kmh").alias("max_speed"),
    ).show(truncate=False)
    threshold_rows = []
    for threshold in speed_thresholds:
        threshold_rows.append((
            threshold,
            travel_df.where(F.col("speed_from_prev_kmh") < threshold).count(),
            non_travel_df.where(F.col("speed_from_prev_kmh") >= threshold).count(),
        ))
    threshold_summary = spark.createDataFrame(
        threshold_rows,
        ["speed_threshold_kmh", "labeled_travel_below_threshold", "unlabeled_rows_above_threshold"],
    )
    print("Threshold sweep:")
    threshold_summary.show(truncate=False)
    print("Fastest non-travel rows:")
    non_travel_df.where(F.col("speed_from_prev_kmh").isNotNull()).orderBy(F.desc("speed_from_prev_kmh")).select(
        "transaction_id", "card_id", "event_time", "distance_from_prev_km", "hours_since_prev", "speed_from_prev_kmh", "anomaly_type"
    ).show(20, truncate=False)
    print("Slowest labeled impossible-travel rows:")
    travel_df.orderBy(F.asc("speed_from_prev_kmh")).select(
        "transaction_id", "ref_transaction_id", "event_time", "distance_from_prev_km", "hours_since_prev", "speed_from_prev_kmh"
    ).show(20, truncate=False)


In [ ]:
# Cleanup when you are done.

spark.stop()
